In [1]:
# !pip install -Iv datasets==3.6.0
# # the dataset used here builds code and execution of trusted code was removed
# # post v3.6.0, so just reinstall the current version if needed later

In [2]:
import nltk
from nltk.corpus import brown
from datasets import load_dataset

In [3]:
nltk.download('brown')
nltk.download('universal_tagset')

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.


True

In [4]:
corpus = brown.tagged_sents(tagset='universal')
corpus

[[('The', 'DET'), ('Fulton', 'NOUN'), ('County', 'NOUN'), ('Grand', 'ADJ'), ('Jury', 'NOUN'), ('said', 'VERB'), ('Friday', 'NOUN'), ('an', 'DET'), ('investigation', 'NOUN'), ('of', 'ADP'), ("Atlanta's", 'NOUN'), ('recent', 'ADJ'), ('primary', 'NOUN'), ('election', 'NOUN'), ('produced', 'VERB'), ('``', '.'), ('no', 'DET'), ('evidence', 'NOUN'), ("''", '.'), ('that', 'ADP'), ('any', 'DET'), ('irregularities', 'NOUN'), ('took', 'VERB'), ('place', 'NOUN'), ('.', '.')], [('The', 'DET'), ('jury', 'NOUN'), ('further', 'ADV'), ('said', 'VERB'), ('in', 'ADP'), ('term-end', 'NOUN'), ('presentments', 'NOUN'), ('that', 'ADP'), ('the', 'DET'), ('City', 'NOUN'), ('Executive', 'ADJ'), ('Committee', 'NOUN'), (',', '.'), ('which', 'DET'), ('had', 'VERB'), ('over-all', 'ADJ'), ('charge', 'NOUN'), ('of', 'ADP'), ('the', 'DET'), ('election', 'NOUN'), (',', '.'), ('``', '.'), ('deserves', 'VERB'), ('the', 'DET'), ('praise', 'NOUN'), ('and', 'CONJ'), ('thanks', 'NOUN'), ('of', 'ADP'), ('the', 'DET'), ('City

In [5]:
corpus[0]

[('The', 'DET'),
 ('Fulton', 'NOUN'),
 ('County', 'NOUN'),
 ('Grand', 'ADJ'),
 ('Jury', 'NOUN'),
 ('said', 'VERB'),
 ('Friday', 'NOUN'),
 ('an', 'DET'),
 ('investigation', 'NOUN'),
 ('of', 'ADP'),
 ("Atlanta's", 'NOUN'),
 ('recent', 'ADJ'),
 ('primary', 'NOUN'),
 ('election', 'NOUN'),
 ('produced', 'VERB'),
 ('``', '.'),
 ('no', 'DET'),
 ('evidence', 'NOUN'),
 ("''", '.'),
 ('that', 'ADP'),
 ('any', 'DET'),
 ('irregularities', 'NOUN'),
 ('took', 'VERB'),
 ('place', 'NOUN'),
 ('.', '.')]

In [6]:
len(corpus)

57340

In [7]:
import json

In [8]:
# will need to make sure it works with say, 100 texts before doing it with the whole thing
treated_data = []
treated_data.append({"a": 1, "b" : 2})
treated_data.append({"c": 3, "d" : 4})
# treated_data

json_lines = [json.dumps(l) for l in treated_data]
json_data = '\n'.join(json_lines)

with open('data_test.jsonl', 'w') as file:
    file.write(json_data)

# with open('data_test2.json', 'w') as file:
#     file.write(json_data)

In [9]:
# the code above lets me match the wanted output format, but it's still weird
# that it handles things like that. Oddly enough, json gives me an error when
# reading the output, jsonl preserves the line by line structure with no errors

## let's try to do the algorithm to convert the corpus to the wanted format

treated_data = []

def treatment(sentence):
    # initialize the list to append and the inner lists
    treated_dict = {}
    inputs = []
    targets = []

    # add the contents into the input/target lists
    for tup in sentence:
        inputs.append(tup[0])
        targets.append(tup[1])

    treated_dict["inputs"] = inputs
    treated_dict["targets"] = targets

    return treated_dict

for sentence in corpus:
    treated_data.append(treatment(sentence))


In [10]:
# sanity check to verify all inner lists per dict have same length
cpt_ok = 0
cpt_ng = 0

for i in range(len(treated_data)):
    if (len(treated_data[i]['inputs']) == len(treated_data[i]['targets'])):
        cpt_ok += 1
    else:
        cpt_ng += 1

print(f'{len(treated_data)} texts treated, {cpt_ok} OK & {cpt_ng} NG')

57340 texts treated, 57340 OK & 0 NG


In [11]:
# all's good so we just need to store the data now
json_lines = [json.dumps(l) for l in treated_data]
json_data = '\n'.join(json_lines)

with open('data.jsonl', 'w') as file:
    file.write(json_data)


In [12]:
data = load_dataset("json", data_files="data.jsonl")

Generating train split: 0 examples [00:00, ? examples/s]

In [13]:
# notice how above we imported it and got only a train set
# we need to split the dataset in train and text subsets

split_data = data['train'].train_test_split(test_size=0.2)

In [14]:
split_data

DatasetDict({
    train: Dataset({
        features: ['inputs', 'targets'],
        num_rows: 45872
    })
    test: Dataset({
        features: ['inputs', 'targets'],
        num_rows: 11468
    })
})

In [15]:
split_data["train"][0]

{'inputs': ['He',
  'supervised',
  'the',
  'cleanups',
  'and',
  'handled',
  'the',
  'shipments',
  'of',
  'raw',
  'gold',
  'which',
  'each',
  'week',
  'went',
  'out',
  'to',
  'San',
  'Francisco',
  '.'],
 'targets': ['PRON',
  'VERB',
  'DET',
  'NOUN',
  'CONJ',
  'VERB',
  'DET',
  'NOUN',
  'ADP',
  'ADJ',
  'NOUN',
  'DET',
  'DET',
  'NOUN',
  'VERB',
  'PRT',
  'ADP',
  'NOUN',
  'NOUN',
  '.']}

In [16]:
from transformers import AutoTokenizer

# try BERT later as homework
checkpoint = 'distilbert-base-cased'
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [17]:
labels = list(set([x for l in split_data['train']['targets'] for x in l]))
labels

['ADV',
 '.',
 'ADJ',
 'X',
 'NOUN',
 'PRT',
 'DET',
 'ADP',
 'CONJ',
 'PRON',
 'NUM',
 'VERB']

In [18]:
id2label = dict(enumerate(labels))
label2id = {v:k for k,v in id2label.items()}

In [19]:
label2id

{'ADV': 0,
 '.': 1,
 'ADJ': 2,
 'X': 3,
 'NOUN': 4,
 'PRT': 5,
 'DET': 6,
 'ADP': 7,
 'CONJ': 8,
 'PRON': 9,
 'NUM': 10,
 'VERB': 11}

In [20]:
id2label

{0: 'ADV',
 1: '.',
 2: 'ADJ',
 3: 'X',
 4: 'NOUN',
 5: 'PRT',
 6: 'DET',
 7: 'ADP',
 8: 'CONJ',
 9: 'PRON',
 10: 'NUM',
 11: 'VERB'}

In [21]:
idx = 0
t = tokenizer(split_data["train"][idx]["inputs"], is_split_into_words=True)
t

{'input_ids': [101, 1124, 14199, 1103, 4044, 17210, 1105, 8630, 1103, 25464, 1116, 1104, 7158, 2284, 1134, 1296, 1989, 1355, 1149, 1106, 1727, 2948, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [22]:
type(t)

transformers.tokenization_utils_base.BatchEncoding

In [23]:
t.tokens()

['[CLS]',
 'He',
 'supervised',
 'the',
 'clean',
 '##ups',
 'and',
 'handled',
 'the',
 'shipment',
 '##s',
 'of',
 'raw',
 'gold',
 'which',
 'each',
 'week',
 'went',
 'out',
 'to',
 'San',
 'Francisco',
 '.',
 '[SEP]']

In [24]:
for x,y in zip(t.tokens(), t.word_ids()):
    print(f"{x}\t{y}")

[CLS]	None
He	0
supervised	1
the	2
clean	3
##ups	3
and	4
handled	5
the	6
shipment	7
##s	7
of	8
raw	9
gold	10
which	11
each	12
week	13
went	14
out	15
to	16
San	17
Francisco	18
.	19
[SEP]	None


In [25]:
# the value of i indicates it is the i'th word
# in the input sentence (counting from 0)
t.word_ids()

[None,
 0,
 1,
 2,
 3,
 3,
 4,
 5,
 6,
 7,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 None]

In [26]:
# function to make the labels the same length as the tokenized word list
def align_targets(labels, word_ids):
    aligned_labels = []
    for word in word_ids:
        if word is None:
            # a type of token, like [CLS] or [SEP]
            label = -100

        else:
            # it's a new word
            label = label2id[labels[word]]

        # add the label
        aligned_labels.append(label)

    return aligned_labels

In [27]:
# trying the function
labels = split_data["train"][idx]["targets"]
# labels
word_ids = t.word_ids()
# print(len(labels))
# print(len(word_ids))
aligned_targets = align_targets(labels, word_ids)
aligned_targets

[-100,
 9,
 11,
 6,
 4,
 4,
 8,
 11,
 6,
 4,
 4,
 7,
 2,
 4,
 6,
 6,
 4,
 11,
 5,
 7,
 4,
 4,
 1,
 -100]

In [28]:
aligned_labels = [id2label[t] if t >= 0 else None for t in aligned_targets]
for x, y in zip(t.tokens(), aligned_labels):
    print(f"{y}\t{x}")

None	[CLS]
PRON	He
VERB	supervised
DET	the
NOUN	clean
NOUN	##ups
CONJ	and
VERB	handled
DET	the
NOUN	shipment
NOUN	##s
ADP	of
ADJ	raw
NOUN	gold
DET	which
DET	each
NOUN	week
VERB	went
PRT	out
ADP	to
NOUN	San
NOUN	Francisco
.	.
None	[SEP]


In [29]:
# # testing with a fake input
# words = [
#     '[CLS]', 'Ger', '##man', 'call', 'to', 'boycott', 'Micro', '##soft', '[SEP]'
# ]
# word_ids = [None, 0, 0, 1, 2, 3, 4, 4, None]
# labels = [7, 0, 0, 0, 3]
# aligned_targets = align_targets(labels, word_ids)
# aligned_labels = [label_names[t] if t >= 0 else None for t in aligned_targets]
# for x, y in zip(t.tokens(), aligned_labels):
#     print(f"{x}\t{y}")

In [30]:
def tokenize_fn(batch):
    # tokenize the input sequence first
    # this populates input_ids, attention_mpask, etc.
    tokenized_inputs = tokenizer(
        batch["inputs"], truncation=True, is_split_into_words=True
    )

    labels_batch = batch["targets"] # original targets
    aligned_labels_batch = [] # aligned targets
    for i, labels in enumerate(labels_batch):
        word_ids = tokenized_inputs.word_ids(i)
        aligned_labels_batch.append(align_targets(labels, word_ids))

    # recall: the 'target' must be stored in the key called 'labels'
    tokenized_inputs["labels"] = aligned_labels_batch

    return tokenized_inputs

In [31]:
# we want to remove these from the model inputs
# they're neither inputs nor targets
split_data['train'].column_names

['inputs', 'targets']

In [32]:
tokenized_datasets = split_data.map(
    tokenize_fn,
    batched=True,
    remove_columns=split_data["train"].column_names,
)

Map:   0%|          | 0/45872 [00:00<?, ? examples/s]

Map:   0%|          | 0/11468 [00:00<?, ? examples/s]

In [33]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 45872
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 11468
    })
})

In [34]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [35]:
# https://stackoverflow.com/questions/11264684/flatten-list-of-lists
def flatten(list_of_lists):
  flattened = [val for sublist in list_of_lists for val in sublist]
  return flattened

In [36]:
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

def compute_metrics(logits_and_labels):
  logits, labels = logits_and_labels
  preds = np.argmax(logits, axis=-1)

  # remove -100 from labels and predictions
  labels_jagged = [[t for t in label if t != -100] for label in labels]

  # do the same for predictions whenever true label is -100
  preds_jagged = [[p for p, t in zip(ps, ts) if t != -100] \
      for ps, ts in zip(preds, labels)
  ]

  # flatten labels and preds
  labels_flat = flatten(labels_jagged)
  preds_flat = flatten(preds_jagged)

  acc = accuracy_score(labels_flat, preds_flat)
  f1 = f1_score(labels_flat, preds_flat, average='macro')

  return {
    'f1': f1,
    'accuracy': acc,
  }



In [37]:
labels = [[-100, 0, 0, 1, 2, 1, -100]]
logits = np.array([[
  [0.8, 0.1, 0.1],
  [0.8, 0.1, 0.1],
  [0.8, 0.1, 0.1],
  [0.1, 0.8, 0.1],
  [0.1, 0.8, 0.1],
  [0.1, 0.8, 0.1],
  [0.1, 0.8, 0.1],
]])
compute_metrics((logits, labels))

{'f1': 0.6, 'accuracy': 0.8}

In [38]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    checkpoint,
    id2label=id2label,
    label2id=label2id,
)

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [39]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    "distilbert-finetuned-ner",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=2,
)

In [40]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)


In [41]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.050378,0.045218,0.962486,0.987026
2,0.021423,0.041871,0.969376,0.988911


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=11468, training_loss=0.048077243855492675, metrics={'train_runtime': 693.4678, 'train_samples_per_second': 132.297, 'train_steps_per_second': 16.537, 'total_flos': 1188053559099072.0, 'train_loss': 0.048077243855492675, 'epoch': 2.0})

In [42]:
trainer.save_model('my_saved_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [43]:
from transformers import pipeline

pipe = pipeline(
  "token-classification",
  model='my_saved_model',
  device=0,
)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [44]:
s = "Bill Gates was the CEO of Microsoft in Seattle, Washington."
pipe(s)

[{'entity': 'NOUN',
  'score': np.float32(0.99987555),
  'index': 1,
  'word': 'Bill',
  'start': 0,
  'end': 4},
 {'entity': 'NOUN',
  'score': np.float32(0.9998822),
  'index': 2,
  'word': 'Gates',
  'start': 5,
  'end': 10},
 {'entity': 'VERB',
  'score': np.float32(0.99992025),
  'index': 3,
  'word': 'was',
  'start': 11,
  'end': 14},
 {'entity': 'DET',
  'score': np.float32(0.9999523),
  'index': 4,
  'word': 'the',
  'start': 15,
  'end': 18},
 {'entity': 'NOUN',
  'score': np.float32(0.9998648),
  'index': 5,
  'word': 'CEO',
  'start': 19,
  'end': 22},
 {'entity': 'ADP',
  'score': np.float32(0.99986875),
  'index': 6,
  'word': 'of',
  'start': 23,
  'end': 25},
 {'entity': 'NOUN',
  'score': np.float32(0.99993396),
  'index': 7,
  'word': 'Microsoft',
  'start': 26,
  'end': 35},
 {'entity': 'ADP',
  'score': np.float32(0.999772),
  'index': 8,
  'word': 'in',
  'start': 36,
  'end': 38},
 {'entity': 'NOUN',
  'score': np.float32(0.99993706),
  'index': 9,
  'word': 'Seat